# Lightningfish — HN backtests on a free GPU

[Lightningfish](https://github.com/rajul-kk/LightningFish) is a domain-agnostic
multi-agent opinion simulation engine. This notebook runs its Hacker News
reception backtests on Kaggle, where a T4 or P100 holds the model in VRAM.

The same runs on a loaded 16 GB CPU box take 6–10 hours and thrash the pagefile.
Here they are minutes.

## What is being tested

Whether a population of LLM agents reading a submission predicts its reception
better than trivial references. Every run reports a **baseline ladder**, and the
simulation must beat *every* rung to mean anything:

| Rung | Reference | Controls for |
|---|---|---|
| 0 | majority class | degenerate data |
| 1 | `naive` | metadata signal, no model |
| 2 | `single_llm` | **the model alone, without the multi-agent machinery** |
| 3 | the simulation | — |

Rung 2 is the one that matters. Beating a heuristic shows the model knows
something; only beating one raw call shows the agents, rounds and herding
contribute anything.

## Results so far — expect negatives

| Run | Majority | Best baseline | Sim | Verdict |
|---|---|---|---|---|
| points, submission-only | 50% | 69% (karma) | 47% | fails |
| points, +2h comments | 50% | **86%** (comment count) | — | baseline jumped |
| points, blind subgroup | 73% | 73% | **32%** | fails badly |

The headline finding is that HN reception is predicted by **who posts** and
**whether anyone replies early** — not by anything the model reads in the
submission. On the blind subgroup (stories with no early comments, where content
is the only signal) the simulation predicted "viral" on 21 of 22 stories and
scored 32% against a 73% constant guess.

You are re-running a **reproduction of negative results**. If your numbers look
dramatically better, suspect a bug before celebrating.

See [METHODOLOGY.md](https://github.com/rajul-kk/LightningFish/blob/main/METHODOLOGY.md)
for the protocol and [ARCHITECTURE.md](https://github.com/rajul-kk/LightningFish/blob/main/ARCHITECTURE.md)
section 10 for the full findings log.

---
## Before you run

**Sidebar settings:** Accelerator → **GPU**, Internet → **On**. Internet is
required to install Ollama and pull the model; enabling it is a one-time account
phone-verification.

**Which accelerator?** Either works. qwen2.5:7b Q4 is ~4.7 GB and fits on one
16 GB card.

- **P100** — 732 GB/s bandwidth, faster at *token generation*
- **T4 x2** — tensor cores, faster at *prompt processing*

This workload is prefill-heavy (long prompts, 16-token outputs), which mildly
favours the T4, but expect them within ~1.5x. **Ollama will not use both T4s** —
the model fits on one, so the second sits idle. Cell 4 measures actual
throughput in under a minute, which beats speculating.

**Never paste API keys here.** This runs a local model and needs none. For a
Claude-backed run use Kaggle *Secrets*, never an inline string.

---
## 1. Configuration

Every knob lives here. Nothing below needs editing.

In [ ]:
import os

MODEL      = "qwen2.5:7b"   # must fit in VRAM; 7B Q4 is ~4.7 GB
N_AGENTS   = 24             # population size per simulation
N_ROUNDS   = 4              # rounds of opinion updating
PULL_LIMIT = 40             # stories for the standard runs

# Large-n run (optional, section 6). At ~26 LLM calls/event, budget roughly
# 50s/event and confirm against the benchmark in section 3 before committing.
SCALE_LIMIT     = 200
SCALE_BUDGET_MIN = 420      # stop simulating past this and score what exists

# Bounds any single request so one wedged call cannot stall a long run.
os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_N_AGENTS"] = str(N_AGENTS)
os.environ["LIGHTNINGFISH_N_ROUNDS"] = str(N_ROUNDS)
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"
os.environ["PYTHONUNBUFFERED"] = "1"

print(f"{MODEL}  |  {N_AGENTS} agents x {N_ROUNDS} rounds  |  {PULL_LIMIT} stories")

---
## 2. Setup

Install Ollama, start it, pull the model, and **verify it is actually on the
GPU**. That last check matters: landing on CPU is silent and looks identical to
working, except everything takes 50x longer.

Each step fails loudly rather than letting a broken install surface three cells
later as a puzzling `FileNotFoundError`.

In [ ]:
# The current Ollama installer extracts with zstd, which the Kaggle image does
# not ship. Without it the install fails but the cell keeps going, and you only
# find out later via a confusing "No such file or directory: 'ollama'".
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!zstd --version || echo "WARNING: zstd still missing - the install below will fail"

!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import shutil, subprocess, time, requests

if shutil.which("ollama") is None:
    raise RuntimeError(
        "ollama binary not found after install. Scroll up for the installer's "
        "own error. Two usual causes: zstd missing (the cell above handles it, "
        "check it succeeded), or Internet not enabled in the sidebar."
    )

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama installed but the server did not come up")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama pull {MODEL}

# Force a load so /api/ps reports placement. keep_alive=-1 pins the model in
# VRAM; otherwise it unloads after 5 idle minutes and reloads mid-run.
requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
    timeout=600,
)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram_gb = m.get("size_vram", 0) / 1e9
    print(f"{m['name']}: {vram_gb:.2f} GB in VRAM")
    assert vram_gb > 0, (
        "Model is on CPU, not GPU. Check the accelerator is enabled in the "
        "sidebar - running this on CPU is pointless."
    )
print("GPU inference confirmed")

In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest

import sys

os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine and HN suites only. The finance/service tests pull yfinance, praw,
# edgar, fastapi, modal and psycopg - none of which an HN run touches.
!python -m pytest tests/core tests/hn -q 2>&1 | tail -3

---
## 3. Throughput check

Measures real seconds-per-call before you commit to a long run. On a starved CPU
box this was ~27s; on GPU expect well under a second. If you see double digits,
revisit the GPU assertion above.

In [ ]:
import time

from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.",
                         "Rate this: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3

print(f"{per_call:.2f}s per short call")
print(f"~{per_call * 26:.0f}s per event  (~26 LLM calls: 2 posts + 4 opinions "
      f"per round, plus baselines)")
print(f"estimate for {PULL_LIMIT} events: {per_call * 26 * PULL_LIMIT / 60:.0f} min")

---
## 4. Submission-only run

Pulls class-balanced settled stories — half above the high points threshold,
half below the low one, so the majority-class rung cannot be trivially high —
and scores the simulation against the karma heuristic, one raw model call, and
the majority class.

This also populates the cache that section 5 reuses, so **run it first**.

In [ ]:
!python -m tests.integration.run_backtest hn {PULL_LIMIT} 2>&1 | tee /kaggle/working/hn.log

---
## 5. Early comments — paired

The same stories, re-seeded with every comment posted in their first 2 hours.
Story ids come from the cache above, so the comparison is **paired** rather than
a different sample, and the base run's ground-truth measurements are reused
(HN points keep accruing, so re-measuring would silently unpair the runs).

The ladder gains a `naive_early` rung predicting from the comment **count**. The
simulation only earns a result by beating it — which would mean it is reading
the comment *text*, not just noticing comments exist.

Because the seed now contains post-submission information, these numbers are
**not comparable** to section 4.

In [ ]:
!python -m tests.integration.run_backtest hn-early 2>&1 | tee /kaggle/working/hn_early.log

### The blind subgroup

Restricted to stories that drew *no* early discussion — the only place the
comment-count baseline carries no information and the simulation must read
submission content to beat it.

This is the sharpest test in the notebook, and the one the simulation currently
fails hardest (32% against a 73% constant guess).

In [ ]:
!python -m tests.integration.run_backtest hn-early blind 2>&1 | tee /kaggle/working/hn_blind.log

---
## 6. Optional — large-n run

**Why bother:** sample size is the binding constraint on every claim in the
findings log. At n≈22–36 the binomial test cannot register anything short of an
enormous margin, so those runs support *negative* conclusions but could never
support a positive one. To clear a 70% baseline at p<0.05 you need ~85% accuracy
at n=40, but only ~76% at n=400.

A few hundred stories is impractical on CPU and routine here. This section
budgets its time and checkpoints, so a session timeout does not cost everything.

Skip it if you only wanted the reproduction above.

In [ ]:
from lightningfish_core.event_cache import CachingAdapter, EventCache, cached_pull_events
from lightningfish_hn.backtest_events import pull_hn_events
from lightningfish_hn.config import HNCommentsAdapter, HNDomainAdapter

cache = EventCache("hn_stories")
points_adapter = CachingAdapter(HNDomainAdapter(), cache)
comments_adapter = CachingAdapter(HNCommentsAdapter(), cache)

events = cached_pull_events(
    cache, f"hn:points:{SCALE_LIMIT}", lambda: pull_hn_events("points", SCALE_LIMIT)
)
print(f"{len(events)} events available")
print(f"projected: {per_call * 26 * len(events) / 3600:.1f} hours "
      f"(budget is {SCALE_BUDGET_MIN / 60:.1f}h)")

In [ ]:
import json, pathlib

from lightningfish_core.engine import SimulationEngine

engine = SimulationEngine(points_adapter, model=f"ollama:{MODEL}")
pairs, t0, budget_s = [], time.time(), SCALE_BUDGET_MIN * 60

for i, ev in enumerate(events, 1):
    agents = points_adapter.build_personas(N_AGENTS)
    pairs.append((ev, engine.run(ev.seed, agents, n_rounds=N_ROUNDS)))

    if i % 10 == 0 or i == len(events):
        elapsed = time.time() - t0
        rate = elapsed / i
        print(f"{i}/{len(events)}  {rate:.1f}s/event  "
              f"elapsed {elapsed/60:.1f}m  eta {rate*(len(events)-i)/60:.1f}m", flush=True)
        pathlib.Path("/kaggle/working/progress.json").write_text(json.dumps({
            "done": i, "total": len(events), "seconds_per_event": rate,
            "finals": [(e.event_id, r.trajectory[-1] if r.trajectory else 0.0)
                       for e, r in pairs],
        }, indent=2))

    if time.time() - t0 > budget_s:
        print(f"\nbudget reached at {i}/{len(events)} - scoring what we have")
        break

print(f"simulated {len(pairs)} events in {(time.time()-t0)/60:.1f} min")

In [ ]:
from lightningfish_core.backtest import llm_baseline, score_precomputed, sign

for label, adapter in (("points / reception", points_adapter),
                       ("num_comments / engagement", comments_adapter)):
    report = score_precomputed(adapter, pairs, baselines={
        "naive": lambda e, a=adapter: sign(a.naive_prediction(e.seed)),
        "single_llm": llm_baseline(adapter, engine),
    })
    print(f"\n=== hn {label} ===")
    print(f"  n={report.n_events}  sim={report.sim_accuracy:.1%}  "
          f"majority={report.majority_class_accuracy:.1%}")
    for name, acc in report.baseline_accuracy.items():
        verdict = "PASS" if report.beats_baselines[name] else "FAIL"
        print(f"  vs {name:<12} {acc:>6.1%}   {verdict}")
    print(f"  p_value_vs_best = {report.p_value_vs_best:.4f}")
    print(f"  parse_rate={report.mean_parse_success_rate:.2f}  "
          f"low_confidence={report.low_confidence_events}  skipped={report.skipped}")

---
## 7. Save results

In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/*.log /kaggle/working/cache 2>/dev/null

Everything under `/kaggle/working` is kept as notebook output. Saving the cache
lets a later session re-score without re-fetching — and, because HN points keep
accruing, preserves the *original measurements* so future comparisons stay
paired.

---
## How to read the output

1. **Confidence first.** If `low_confidence` is set or `parse_rate` < 0.8, stop.
   The result reflects malformed model output, not dynamics.
2. **`beats_baselines` must be PASS on every rung**, including `single_llm`, and
   the accuracy must exceed the majority class. Three out of four is a failure.
3. **`p_value_vs_best`** tests against the *best* rung, not chance. High accuracy
   with p > 0.05 is not a result.
4. **Expect the simulation to lose.** That is the current finding, and it is
   reported rather than hidden. A protocol that only publishes wins is not a
   protocol.

If you scale past a few hundred events, the interesting question stops being
"does it win" and becomes whether 32% on the blind subgroup is genuinely that
bad or partly small-sample noise. Either answer belongs in the findings log.